<a href="https://colab.research.google.com/github/RaquelHernanz/BachelorsThesis_SyntheticClinicalData/blob/master/NOTEBOOK1_BTSD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Notebook 1 — Dataset Transformations prior to EDA**

- **Author:** Raquel Hernanz Hernández
- **Supervisors:** José María Herrera and Guillermo José Ortega
- **Degree:** Biomedical Engineering  
- **Project:** Bachelor Thesis — *Generation and Validation of Synthetic Data from a Hospital Emergency Department*

## Role in the pipeline

| Notebook | Purpose |
|----------|---------|
| **NB1 (this notebook)** | Data loading, cleaning, column removal, binary re-encoding, export |
| NB2 | Exploratory Data Analysis (distributions, clustering, feature screening) |
| NB3 | Outlier detection and clinical plausibility audit |
| NB4 | VIF analysis, logistic regression, clinical risk scoring system |

## Objectives

**Notebook 1 (NB1)** transforms the raw clinical dataset into a clean, analysis-ready file by:

1. Standardising column names and removing structural artefacts from Excel export.
2. Normalising heterogeneous missing-value tokens into a single representation (`NaN`).
3. Quantifying missingness at the attribute and instance levels to guide column/row decisions.
4. Removing columns that will not be used in downstream notebooks (hospital/post-ED variables, redundant text fields, high-missingness attributes).
5. Re-encoding binary variables from the clinical `{1 = Yes, 2 = No}` convention to the modelling-standard `{1, 0}`.
6. Exporting a single definitive CSV consumed by NB2, NB3, and NB4.

## Input / Output

| | File | Description |
|---|---|---|
| **Input** | `raw_dataset.xlsx` | Raw Excel export from La Princesa ED (~2376 episodes, ~97 columns) |
| **Output** | `dataset_FINAL.csv` | Cleaned dataset with prehospital variables + mortality outcomes only |


## **1. Libraries**

Only three main libraries are needed for this notebook, the transformations are purely structural (no modelling, no plotting):

| Library | Role |
|---------|------|
| `pandas` | DataFrame operations: loading Excel, column manipulation, missing-value handling, CSV export |
| `numpy` | Numerical missing-value sentinel (`np.nan`) and ceiling function for threshold computation |
| `re` | Regular expressions for column-name cleaning (prefix stripping) |
| `os` | File system operations: checking file size post-export (`os.path.getsize`) |
| `collections` | `defaultdict` used to track column removal reasons per category (Section 9) |


>**No machine-learning or visualisation imports are required at this stage; those are deferred to NB2–NB4.**

In [ ]:
import pandas as pd
import numpy as np
import re
import os
from collections import defaultdict

## **2. Shared configuration**

A single configuration block defines all paths, constants, and thresholds used throughout the notebook. The same block (with identical `RANDOM_STATE`, `TARGET_COL`, and file paths) is replicated at the top of **NB2**–**NB4** to ensure consistency across the pipeline.

### Parameters

| Parameter | Value | Purpose |
|-----------|-------|---------|
| `RANDOM_STATE` | `42` | Global seed for reproducibility (used in NB2–NB4; declared here for consistency) |
| `TARGET_COL` | `"Mort. 2D"` `"Mort. 7D"` `"Mort. 30D"` | Primary outcomes: 2, 7 and 30-day mortality (binary) |
| `MAX_MISSING_PCT` | `0.15` | Threshold (15 %) above which a column is flagged for removal |
| `RAW_DATA_PATH` | (see code) | Path to the raw Excel file (edit per environment) |
| `CLEAN_DATA_PATH` | (see code) | Path where the cleaned CSV will be saved |
| `ATTR_TABLE_PATH` | (see code) | Path to the attribute reference table (used in NB3 for plausibility ranges) |

Every threshold or path is defined once, documented here, and referenced by name in subsequent cells. If the dataset or environment changes, only this cell needs editing.

In [ ]:
# ══════════════════════════════════════════════════════════════
# SHARED CONFIGURATION (replicated in NB2, NB3, NB4)
# ══════════════════════════════════════════════════════════════

RANDOM_STATE   = 42
TARGET_COL     = "Mort. 30D" #Just the default
ALL_MORT_COLS  = ["Mort. 2D", "Mort. 7D", "Mort. 30D"]
MAX_MISSING_PCT = 0.15   # 15 % threshold for column removal

# --- File paths (edit per environment) ---
RAW_DATA_PATH   = "/content/drive/MyDrive/Colab Notebooks/TFG/Francisco_Base de datos 12452209 (La Princesa).xlsx"
CLEAN_DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/TFG/dataset_FINAL.csv"
ATTR_TABLE_PATH = "/content/drive/MyDrive/Colab Notebooks/TFG/TABLA_ATRIBUTOS_ACTUALIZADO.xlsx"

# --- Colab Google Drive mount (no-op if already mounted or not in Colab) ---
try:
    from google.colab import drive
    from pathlib import Path
    if not Path("/content/drive").exists():
        drive.mount("/content/drive")
except Exception:
    pass

Mounted at /content/drive


## **3. Data loading**

1. Read the raw Excel file into a pandas DataFrame (`df_raw`).
2. Create a working copy (`df`) to preserve the original for comparison.
3. Print the shape and first rows as a sanity check.

### Input

| Parameter | Description |
|-----------|-------------|
| `RAW_DATA_PATH` | Absolute path to the `.xlsx` file exported from the hospital information system |

### Output

| Variable | Type | Description |
|----------|------|-------------|
| `df_raw` | `pd.DataFrame` | Immutable copy of the raw data (never modified) |
| `df` | `pd.DataFrame` | Working copy on which all transformations are applied |

> Keeping `df_raw` untouched allows before/after comparisons at any point in the notebook. The working copy `df` accumulates all cleaning steps sequentially.

In [ ]:
df_raw = pd.read_excel(RAW_DATA_PATH)

print(f"Dataset dimensions: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
display(df_raw.head(3))

# Working copy
df = df_raw.copy()

# --- Show all column names ---
print("\nTotal columns:", len(df.columns))

# List (simple)
print("\nColumn names (list):")
print(df.columns.tolist())


Dataset dimensions: 2376 rows × 98 columns


,Fecha,Edad,Sexo,UME,H. Act.,T. Lleg.,H. Asi.,T. Asis.,H. Tra.,T. Trasl,...,DgH #3,Descripción.3,Fecha alta,Días ingreso,Mort. 2D,Mort. 7D,Mort. 30D,Sepsis,Infección,Foco
0,2018-01-31,87,1,1,09:36:00,00:11:00,09:47:00,00:13:00,10:00:00,00:18:00,...,NaN,NaN,2018-01-31,0,1,1,1,1,1,Respiratorio
1,2018-01-29,87,2,1,01:17:00,00:07:00,01:24:00,00:36:00,02:00:00,00:19:00,...,NaN,NaN,2018-02-09,11,2,2,2,2,2,NaN
2,2018-02-03,88,1,1,05:46:00,00:11:00,05:57:00,00:26:00,06:23:00,00:14:00,...,NaN,NaN,2018-02-13,10,2,2,2,2,1,NaN



Total columns: 98

Column names (list):
['Fecha', 'Edad', 'Sexo', 'UME', 'H. Act.', 'T. Lleg.', 'H. Asi.', 'T. Asis.', 'H. Tra.', 'T. Trasl', 'H. Lle.', 'Motivo L.', 'Destino', 'Servicio', 'FR', 'SpO2', 'O2', 'TAS', 'TAD', 'TAM', 'FC', 'TT', 'GCS.O', 'GCS.V', 'GCS.M', 'GCS', 'Lactato', 'Glucemia', 'Ritmo', 'ST', 'O2 sup.', 'FiO2', 'Gafas', 'Venturi', 'Resrv.', 'Nebul.', 'VNI', 'IOT', 'VAD', 'VM', 'MAVA', 'TTE', 'Med #1', 'Med #2', 'Med #3', 'Med #4', 'Med #5', 'Dg #1', 'Descripción', 'Dg #2', 'Dg #3', 'Triaje', 'FRH', 'SpO2H', 'O2H', 'TASH', 'TADH', 'TAMH', 'FCH', 'TTH', 'GCS.O.H', 'GCS.V.H', 'GCS.M.H', 'GCS.H', 'TAC', 'ECO', 'Endosc.', 'Cirugía', 'Creati.', 'Bilirrub.', 'PLQ', 'Hct.', 'Leuc', 'LAC', 'pH', 'PCR', 'PCT', 'Dime. D', 'BNP', 'Trop.', 'INTERC.', 'Hospital.', 'UCI', 'Días UCI', 'DgH #1', 'Descripción.1', 'DgH #2', 'Descripción.2', 'DgH #3', 'Descripción.3', 'Fecha alta', 'Días ingreso', 'Mort. 2D', 'Mort. 7D', 'Mort. 30D', 'Sepsis', 'Infección', 'Foco']


## **4. Duplicate episode detection and leakage mitigation**

Electronic health records from emergency departments frequently contain duplicate
entries arising from administrative re-registrations, system transfers, or data
extraction artefacts. In supervised learning pipelines, retaining duplicates
without controlling their allocation across partitions introduces data leakage:
identical feature vectors in both training and test sets produce artificially
optimistic performance estimates.

### Detection

Duplicate rows are identified using exact matching across all feature columns
via `pandas.DataFrame.duplicated(keep=False)`, which flags all copies of a
repeated episode rather than preserving one.
### Retention decision

Duplicate removal was **not performed** despite the leakage risk. Given the low count of positive outcome cases, such as `Mort. 2D (n = 114)`,
elimination would reduce positive cases by a non-negligible proportion,
compromising statistical power in an already imbalanced dataset. Another issue is the lack of unique identifiers execept for the `Fecha` attribute after the anonimization. Therefore, we cannot guarantee whether or not instances are true duplicates.

In [ ]:
# ── Duplicate rows: pre- and post-correction outcome distribution analysis ────

"""
Duplicate rows are identified using exact matching across all feature columns
via `pandas.DataFrame.duplicated(keep=False)`, which flags all copies of a
repeated episode rather than preserving one.
"""

OUTCOME_COLS = ['Mort. 2D', 'Mort. 7D', 'Mort. 30D']

def _prev_stats(subset: pd.DataFrame, col: str) -> tuple:
    """Return prevalence string and event count, filtered to valid {0, 1} values.

    Parameters
    ----------
    subset : pd.DataFrame
        Subset of the DataFrame (e.g., duplicates only or unique rows only).
    col : str
        Name of the binary outcome column to analyse.

    Returns
    -------
    tuple[str, int]
        (prevalence_str, n_events): percentage of events as a formatted
        string, and the absolute event count.
    """
    y = pd.to_numeric(subset[col], errors="coerce").astype("Int64")
    valid    = y.isin([0, 1])
    n_valid  = int(valid.sum())
    n_events = int(y[valid].sum())
    prev_str = f"{n_events/n_valid*100:.2f}" if n_valid > 0 else "N/A"
    return prev_str, n_events

def _dup_summary(dataframe: pd.DataFrame, label: str) -> int:
    """Print duplicate counts and outcome prevalence for a given DataFrame.

    Parameters
    ----------
    dataframe : pd.DataFrame
        Input DataFrame to analyse for duplicate rows.
    label : str
        Descriptive label printed in the section header
        (e.g., ``'PRE-CORRECTION'``).

    Returns
    -------
    int
        Number of duplicated rows detected in the DataFrame.
    """
    dup_mask  = dataframe.duplicated(keep=False)
    df_dup    = dataframe[dup_mask]
    df_unique = dataframe[~dup_mask]
    n_total   = len(dataframe)
    n_dup     = int(dup_mask.sum())

    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    print(f"  Total rows      : {n_total}")
    print(f"  Duplicated rows : {n_dup} ({n_dup/n_total*100:.2f} %)")
    print(f"  Unique rows     : {n_total - n_dup}")

    records = []
    for col in OUTCOME_COLS:
        prev_all,  n_all  = _prev_stats(dataframe, col)
        prev_dup,  n_dup_ = _prev_stats(df_dup,    col)
        prev_uniq, n_uniq = _prev_stats(df_unique,  col)
        records.append({
            "Outcome"         : col,
            "Overall (%)"     : prev_all,
            "Duplicates (%)": prev_dup,
            "Unique (%)"      : prev_uniq,
            "N duplicates"    : n_dup_,
            "N unique"        : n_uniq,
        })

    summary = pd.DataFrame(records).set_index("Outcome")
    print("\n  Outcome prevalence by duplicate status (valid {0,1} rows only):")
    display(summary)
    return n_dup

# ── 1. PRE-CORRECTION ─────────────────────────────────────────────────────────
n_dup_pre = _dup_summary(df, "PRE-CORRECTION (raw mortality flags)")


  PRE-CORRECTION (raw mortality flags)
  Total rows      : 2376
  Duplicated rows : 214 (9.01 %)
  Unique rows     : 2162

  Outcome prevalence by duplicate status (valid {0,1} rows only):


,Overall (%),Duplicates (%),Unique (%),N duplicates,N unique
Outcome,,,,,
Mort. 2D,100.00,100.00,100.00,12,102
Mort. 7D,100.00,100.00,100.00,18,154
Mort. 30D,100.00,100.00,100.00,28,231


## **5. Column name standardisation**

1. **Strip whitespace** — remove leading/trailing spaces from every column name (common Excel artefact).
2. **Remove numeric prefixes** — some exports produce names like `"001 - Edad"`; the regex `^\s*\d+\s*-\s*` strips the prefix, leaving `"Edad"`.
3. **Remove ghost columns** — columns whose names start with `"Unnamed"` (pandas auto-generated) or whose values are entirely `NaN` (empty columns introduced during export).

> Inconsistent column names cause silent downstream failures: a column named `" Edad"` (with a leading space) will not match a hardcoded reference to `"Edad"`. Cleaning names at the source prevents this class of bugs from propagating to NB2-NB4.

### Parameters

| Parameter | Value | Purpose |
|-----------|-------|---------|
| Prefix regex | `^\s*\d+\s*-\s*` | Matches patterns like `"001 - "`, `" 23-"`, etc. |

### Output

| Variable | Description |
|----------|-------------|
| `df` | DataFrame with clean column names, ghost columns removed |
| Console output | Number of dropped columns, new shape |

In [ ]:
# 1) Strip whitespace
df.columns = [str(c).strip() for c in df.columns]

# 2) Remove numeric prefixes (e.g. "001 - Edad" → "Edad")
def strip_prefix(col: str) -> str:
    """Remove leading numeric prefixes from column names.

    Parameters
    ----------
    col : str
        Original column name, potentially with a prefix like '001 - '.

    Returns
    -------
    str
        Cleaned column name without the numeric prefix.
    """
    return re.sub(r"^\s*\d+\s*-\s*", "", str(col)).strip()

df.columns = [strip_prefix(c) for c in df.columns]

# 3) Remove ghost / empty columns
unnamed_cols = [c for c in df.columns if c.startswith("Unnamed")]
all_nan_cols = df.columns[df.isna().all()].tolist()
ghost_cols   = sorted(set(unnamed_cols + all_nan_cols))

if ghost_cols:
    df = df.drop(columns=ghost_cols)
    print(f"Removed {len(ghost_cols)} ghost/empty columns: {ghost_cols}")
else:
    print("No ghost/empty columns found.")

print(f"Shape after name cleaning: {df.shape[0]} rows × {df.shape[1]} columns")

No ghost/empty columns found.
Shape after name cleaning: 2376 rows × 98 columns


## **6. Missing-value token normalisation**

Clinical datasets frequently encode missing values using heterogeneous textual tokens rather than leaving cells empty. Common examples include `"NA"`, `"N/A"`, `"NULL"`, `"-"`, and whitespace strings. This step replaces all known tokens with `np.nan` so that **Pandas**' missing-value API (`.isna()`, `.dropna()`, etc.) works consistently throughout the pipeline.

### Parameters

| Parameter | Description |
|-----------|-------------|
| `missing_tokens` | List of string patterns to replace with `np.nan` |

> Possible values: `""`, `" "`, `"\t"`, `"\n"`, `"NA"`, `"N/A"`, `"n/a"`, `"NaN"`, `"nan"`, `"NULL"`, `"null"`, `"-"`, `"--"`.

> Without this step, a column containing both `NaN` and the string `"NA"` would report misleading missingness statistics (the string `"NA"` would not be counted as missing). Normalising early ensures that all subsequent analyses operate on a single, unambiguous missing-value representation.

### Output

| Variable | Description |
|----------|-------------|
| `df` | DataFrame with all heterogeneous tokens replaced by `np.nan` |

In [ ]:
missing_tokens = [
    "", " ", "  ", "\t", "\n",
    "NA", "N/A", "n/a", "NaN", "nan",
    "NULL", "null", "-", "--"
]

df = df.replace(missing_tokens, np.nan)

print(f"Columns: {df.shape[1]} | Rows: {df.shape[0]}")
print(f"Total NaN cells after normalisation: {int(df.isna().sum().sum())}")

Columns: 98 | Rows: 2376
Total NaN cells after normalisation: 35419


## **7. Missingness analysis — per attribute (column)**

For each column, compute:
- **`missing_count`**: absolute number of `NaN` cells.
- **`missing_pct`**: percentage of rows with a missing value for that attribute.

> Columns exceeding the threshold `MAX_MISSING_PCT` (15 %) are flagged. This threshold follows common practice in clinical data analysis: attributes with >15 % missingness contribute limited information and risk introducing bias if imputed aggressively. This analysis informs the column removal step (**Section 8**). Rather than relying on a hardcoded drop list, the threshold is applied programmatically so that the pipeline adapts if the source dataset changes.

### Output

| Variable | Type | Description |
|----------|------|-------------|
| `attr_missing` | `pd.DataFrame` | Table with `attribute`, `missing_count`, `missing_pct` sorted by missingness |
| `high_miss_cols` | `list[str]` | Column names exceeding the 15 % threshold |

In [ ]:
"""
Analyzes missingness per attribute (column) in the DataFrame.

This section calculates the absolute count and percentage of missing (NaN) values
for each column in the DataFrame `df`. It then compiles this information into
a new DataFrame `attr_missing`, sorted by missing percentage in descending order.

Columns with a missing percentage exceeding `MAX_MISSING_PCT` (15% in this case)
are identified and stored in the `high_miss_cols` list. This programmatic
approach allows for threshold-based column removal in subsequent steps, adapting
to changes in the dataset's missingness profile.

Outputs:
- `attr_missing` (pd.DataFrame): A table summarizing missing counts and percentages
  for each attribute, sorted by `missing_pct`.
- `high_miss_cols` (list[str]): A list of column names that exceed the defined
  `MAX_MISSING_PCT` threshold.
"""

n_rows, n_cols = df.shape

missing_count = df.isna().sum().astype(int)
missing_pct   = (missing_count / n_rows * 100).round(2)

attr_missing = (
    pd.DataFrame({
        "attribute":     df.columns,
        "missing_count": missing_count.values,
        "missing_pct":   missing_pct.values
    })
    .sort_values("missing_pct", ascending=False)
    .reset_index(drop=True)
)

# Flag columns above threshold
high_miss_cols = attr_missing.loc[
    attr_missing["missing_pct"] > MAX_MISSING_PCT * 100, "attribute"
].tolist()

print(f"Columns with >{MAX_MISSING_PCT*100:.0f}% missingness: {len(high_miss_cols)}")
display(attr_missing[attr_missing["missing_count"] > 0])

Columns with >15% missingness: 20


,attribute,missing_count,missing_pct
0,Med #5,2307,97.10
1,Dg #3,2300,96.80
2,Foco,2266,95.37
3,PCT,2109,88.76
4,Dime. D,2108,88.72
5,Med #4,2096,88.22
6,BNP,2082,87.63
7,Descripción.3,1990,83.75
8,DgH #3,1989,83.71
9,Días UCI,1936,81.48


## **8. Missingness analysis per instance (row)**

For each patient episode (row), compute the number and percentage of missing attributes. This identifies episodes with exceptionally sparse data that might distort downstream analyses. While column-level missingness drives the decision to remove variables, row-level missingness informs whether certain patient records should be flagged or excluded.

### Output

| Variable | Type | Description |
|----------|------|-------------|
| `row_missing` | `pd.DataFrame` | Table with `row_index`, `missing_count`, `missing_pct` for rows with ≥1 NaN |
| `missing_dist` | `pd.DataFrame` | Frequency distribution: how many rows have 0, 1, 2, … missing values |

In [ ]:
"""
Analyzes missingness per instance (row) in the DataFrame.

This section calculates the number and percentage of missing (NaN) values
for each row in the DataFrame `df`. It then compiles this information into
a new DataFrame `row_missing`, sorted by missing count in descending order.

A frequency distribution `missing_dist` is also computed, showing how many
rows have 0, 1, 2, ... missing values, along with their respective percentages.

Finally, it prints a summary including the total NaN cells, the number and
percentage of rows with at least one NaN, and displays the top 10 entries
of the missingness distribution.

Outputs:
- `row_missing` (pd.DataFrame): A table summarizing missing counts and percentages
  for each row, sorted by `missing_count`.
- `missing_dist` (pd.DataFrame): A frequency distribution of the number of
  missing values per row.
"""

miss_per_row = df.isna().sum(axis=1).astype(int)

row_missing = (
    pd.DataFrame({
        "row_index":     df.index.astype(int),
        "missing_count": miss_per_row.values,
        "missing_pct":   (miss_per_row / n_cols * 100).round(2).values
    })
    .sort_values("missing_count", ascending=False)
    .reset_index(drop=True)
)

# Frequency distribution
missing_dist = (
    miss_per_row
    .value_counts()
    .sort_index()
    .rename_axis("missing_count_in_row")
    .reset_index(name="num_rows")
)
missing_dist["pct_rows"] = (missing_dist["num_rows"] / n_rows * 100).round(2)

# Summary
total_nan    = int(df.isna().sum().sum())
rows_any_nan = int((miss_per_row > 0).sum())

print(f"Total NaN cells:          {total_nan}")
print(f"Rows with ≥1 NaN:        {rows_any_nan} ({rows_any_nan/n_rows*100:.1f}%)")
print(f"\nMissingness distribution (top 10):")
display(missing_dist.head(10))

Total NaN cells:          35419
Rows with ≥1 NaN:        2376 (100.0%)

Missingness distribution (top 10):


,missing_count_in_row,num_rows,pct_rows
0,3,2,0.08
1,4,1,0.04
2,5,16,0.67
3,6,15,0.63
4,7,31,1.30
5,8,45,1.89
6,9,74,3.11
7,10,88,3.70
8,11,107,4.50
9,12,158,6.65


## **9. Column removal**

Columns are removed in three categories, each with a distinct rationale:

#### A. *High-missingness columns (programmatic, threshold-based)*

Columns identified in **Section 6** with >15 % missing values. These are removed automatically based on `high_miss_cols`, no hardcoded list.

#### B. *Hospital / post-ED variables (domain-driven)*

Variables measured after the patient arrives at the hospital (e.g., in-hospital vitals `FRH`, `SpO2H`; lab results `Creati.`, `Leuc`; outcomes like `UCI`, `Cirugía`). Including these in a prehospital prediction model would constitute data leakage, the model would have access to information unavailable at the time of the prediction.

These variables are listed explicitly because the decision is clinical, not statistical.

#### C. *Non-informative / redundant text fields*

Diagnosis text descriptions (`Descripción`, `Descripción.1`, etc.), ICD codes (`Dg #1`, `DgH #1`), date or time-stamp text fields (`Fecha`, `H. Act.`, `T. Lleg.`, etc.), and extra medication slots (`Med #1`–`Med #5`) that are sparse and not used in the modelling pipeline.

### Parameters

| Category | Source | Columns removed |
|----------|--------|-----------------|
| A | Missingness (>15 % NaN) | Variable — depends on dataset |
| B | Clinical judgment | 29 hospital/post-ED variables (see code) |
| C | Pipeline design | 17 text/redundant variables (see code) |

---

> The combined effect of these three removals reduces the dataset from 98 columns to the 33 prehospital variables defined in the attribute reference table (`TABLA_ATRIBUTOS_ACTUALIZADO.xlsx`) plus the three mortality outcome columns. This is the feature set for **NB2**–**NB4**.

### Output

| Variable | Type | Description |
|----------|------|-------------|
| `df` | `pd.DataFrame` | DataFrame with only retained columns |
| Console output | | Before/after shape comparison, list of dropped columns per category |

In [ ]:
"""
Removes columns from the DataFrame based on three distinct rationales:

1.  **High-missingness columns (threshold-based)**: Columns identified in Section 6
    with a missingness percentage exceeding `MAX_MISSING_PCT` (15%). These are
    removed programmatically based on the `high_miss_cols` list.

2.  **Hospital / post-ED variables (domain-driven)**: Variables that represent
    information collected after the patient's arrival at the hospital or that
    pose a data leakage risk for prehospital prediction models. This includes
    in-hospital vitals, lab results, and certain outcome variables.

3.  **Non-informative / redundant text fields**: Columns containing textual
    descriptions, sparse codes, date/time fields, or extra medication slots
    that are not utilized in the downstream modeling pipeline.

This section also reports the shape of the DataFrame before and after column
removal, categorizes the dropped columns, identifies any overlaps between the
removal categories, and lists all dropped columns with their respective reasons.

Outputs:
- `df` (pd.DataFrame): The DataFrame with the specified columns removed.
- Console output detailing the removal process and retained columns.
"""

shape_before = df.shape

# --- A. High-missingness columns (from Section 6) ---
drop_A = [c for c in high_miss_cols if c in df.columns]

# (Optional) missingness % for A (computed on the pre-drop df)
miss_pct_A = (df[drop_A].isna().mean() * 100).round(2).to_dict() if drop_A else {}

# --- B. Hospital / post-ED variables (data leakage risk) ---
HOSPITAL_VITALS = [
    "FRH", "SpO2H", "O2H", "TASH", "TADH", "TAMH", "FCH", "TTH",
    "GCS.O.H", "GCS.V.H", "GCS.M.H", "GCS.H",
]
HOSPITAL_LABS = [
    "Creati.", "Bilirrub.", "PLQ", "Hct.", "Leuc",
    "LAC", "pH", "PCR", "PCT", "Dime. D", "BNP", "Trop.",
]
HOSPITAL_OUTCOMES = [
    "TAC", "ECO", "Endosc.", "Cirugía",
    "INTERC.", "Hospital.", "UCI", "Días UCI",
    "DgH #1", "DgH #2", "DgH #3",
    "Fecha alta", "Días ingreso",
    "Sepsis", "Infección", "Foco", "Triaje"
]
drop_B = [c for c in (HOSPITAL_VITALS + HOSPITAL_LABS + HOSPITAL_OUTCOMES) if c in df.columns]

# --- C. Non-informative / redundant text fields ---
TEXT_FIELDS = [
    "Descripción", "Descripción.1", "Descripción.2", "Descripción.3",
    "Dg #1", "Dg #2", "Dg #3","UME","Destino","Servicio","Motivo L."
]
TIME_FIELDS = [
    "H. Act.", "T. Lleg.", "H. Asi.", "T. Asis.",
    "H. Tra.", "T. Trasl", "H. Lle.", "Fecha"
]
EXTRA_MEDS = ["Med #1","Med #2", "Med #3", "Med #4", "Med #5"]
drop_C = [c for c in (TEXT_FIELDS + TIME_FIELDS + EXTRA_MEDS) if c in df.columns]

# --- Build reason map (col -> set of reasons) ---
reasons = defaultdict(set)
for c in drop_A: reasons[c].add("A")
for c in drop_B: reasons[c].add("B")
for c in drop_C: reasons[c].add("C")

# --- Apply all removals ---
all_drops = sorted(reasons.keys())
df = df.drop(columns=all_drops, errors="ignore")

# --- Report ---
print(f"Shape before: {shape_before[0]} × {shape_before[1]}")
print(f"Shape after:  {df.shape[0]} × {df.shape[1]}")

print(f"\nDropped {len(all_drops)} columns total:")
print(f"  A. High missingness (>{MAX_MISSING_PCT*100:.0f}%): {len(drop_A)}")
print(f"  B. Hospital / post-ED:                 {len(drop_B)}")
print(f"  C. Text / redundant:                   {len(drop_C)}")

# Overlap checks
setA, setB, setC = set(drop_A), set(drop_B), set(drop_C)
overlap_AB = sorted(setA & setB)
overlap_AC = sorted(setA & setC)
overlap_BC = sorted(setB & setC)
overlap_ABC = sorted(setA & setB & setC)

if overlap_AB or overlap_AC or overlap_BC or overlap_ABC:
    print("\nOverlaps (same column flagged in multiple categories):")
    if overlap_AB:  print(f"  A ∩ B ({len(overlap_AB)}): {overlap_AB}")
    if overlap_AC:  print(f"  A ∩ C ({len(overlap_AC)}): {overlap_AC}")
    if overlap_BC:  print(f"  B ∩ C ({len(overlap_BC)}): {overlap_BC}")
    if overlap_ABC: print(f"  A ∩ B ∩ C ({len(overlap_ABC)}): {overlap_ABC}")

# Print full list of dropped columns with reasons (and missing % for A)
print("\nDropped columns (with reasons):")
for c in all_drops:
    tag = ",".join(sorted(reasons[c]))
    if "A" in reasons[c]:
        mp = miss_pct_A.get(c, None)
        if mp is not None:
            print(f"  - {c}  [reasons={tag}]  missing%={mp}")
        else:
            print(f"  - {c}  [reasons={tag}]")
    else:
        print(f"  - {c}  [reasons={tag}]")

print(f"\nRetained columns ({df.shape[1]}):")
print(df.columns.tolist())

Shape before: 2376 × 98
Shape after:  2376 × 33

Dropped 65 columns total:
  A. High missingness (>15%): 20
  B. Hospital / post-ED:                 41
  C. Text / redundant:                   24

Overlaps (same column flagged in multiple categories):
  A ∩ B (12): ['BNP', 'Bilirrub.', 'DgH #2', 'DgH #3', 'Dime. D', 'Días UCI', 'Foco', 'LAC', 'PCR', 'PCT', 'Trop.', 'pH']
  A ∩ C (8): ['Descripción.2', 'Descripción.3', 'Dg #2', 'Dg #3', 'Med #2', 'Med #3', 'Med #4', 'Med #5']

Dropped columns (with reasons):
  - BNP  [reasons=A,B]  missing%=87.63
  - Bilirrub.  [reasons=A,B]  missing%=77.06
  - Cirugía  [reasons=B]
  - Creati.  [reasons=B]
  - Descripción  [reasons=C]
  - Descripción.1  [reasons=C]
  - Descripción.2  [reasons=A,C]  missing%=55.3
  - Descripción.3  [reasons=A,C]  missing%=83.75
  - Destino  [reasons=C]
  - Dg #1  [reasons=C]
  - Dg #2  [reasons=A,C]  missing%=73.61
  - Dg #3  [reasons=A,C]  missing%=96.8
  - DgH #1  [reasons=B]
  - DgH #2  [reasons=A,B]  missing%=55.39
 

## **10. Binary variable re-encoding**

The hospital information system encodes binary variables using the clinical convention `{1 = Yes, 2 = No}`. For modelling and interpretability, these are re-encoded to the standard `{1 = Yes, 0 = No}`.

The transformation is applied to all columns identified as `int (binary)` in the attribute reference table:

| Variable | Description |
|----------|-------------|
| `O2` | Oxygen on arrival of the emergency unit |
| `O2 sup.` | Supplementary oxygen therapy |
| `Gafas` | Glasses |
| `Venturi` | Venturi mask |
| `Resrv.` | Reservoir mask |
| `Nebul.` | Nebulisation |
| `VNI` | Non-invasive ventilation (BiPAP/CPAP) |
| `IOT` | Orotracheal intubation |
| `VAD` | Difficult airway |
| `VM` | Mechanical ventilation |
| `MAVA` | Advanced airway management |
| `TTE` | Defibrillation / pacemaker / cardioversion |
| `Mort. 2D` | 2-day mortality |
| `Mort. 7D` | 7-day mortality |
| `Mort. 30D` | 30-day mortality |
| `Sexo` | Biological sex (1 = Male, 2 = Female → 1 = Male, 0 = Female) |

### Function: `recode_binary_12()`

| Parameter | Type | Description |
|-----------|------|-------------|
| `series` | `pd.Series` | Column to recode |
| **Returns** | `pd.Series` | Recoded column with dtype `Int64` (nullable integer) |

### Logic

1. Coerce to numeric (handles any residual string values).
2. Check unique non-null values.
3. If `{1, 2}` → map `{1: 1, 2: 0}`.
4. If already `{0, 1}` → cast to `Int64` (no change).
5. Otherwise → leave unchanged (not a binary column).

This transformation must happen once, at the source (**NB1**) rather than ad hoc in each downstream notebook. The `Int64` nullable integer dtype preserves `NaN` values while keeping the column numeric; standard `int64` would coerce `NaN` to `float64`, causing type inconsistencies in categorical operations.

In [ ]:
def recode_binary_12(series: pd.Series) -> pd.Series:
    """Recode a clinical binary column from {1=Yes, 2=No} to {1=Yes, 0=No}.

    Parameters
    ----------
    series : pd.Series
        Column encoded as {1, 2} or already {0, 1}.

    Returns
    -------
    pd.Series
        Recoded column with nullable Int64 dtype.
    """
    s = pd.to_numeric(series, errors="coerce")
    unique_vals = set(s.dropna().unique())

    if unique_vals.issubset({1, 2}):
        return s.map({1: 1, 2: 0}).astype("Int64")
    elif unique_vals.issubset({0, 1}):
        return s.astype("Int64")
    else:
        return series  # Not a binary column — return unchanged


# Columns to recode (from attribute table: all "int (binaria)" + mortality + Sexo)
BINARY_COLS = [
    "Sexo",
    "O2", "O2 sup.", "Gafas", "Venturi", "Resrv.", "Nebul.",
    "VNI", "IOT", "VAD", "VM", "MAVA", "TTE",
    "Mort. 2D", "Mort. 7D", "Mort. 30D",
]

recoded = []
for col in BINARY_COLS:
    if col not in df.columns:
        continue
    before_vals = sorted(df[col].dropna().unique().tolist())
    df[col] = recode_binary_12(df[col])
    after_vals = sorted(df[col].dropna().unique().tolist())
    if before_vals != after_vals:
        recoded.append(col)

print(f"Recoded {len(recoded)} columns from {{1,2}} → {{1,0}}:")
for col in recoded:
    print(f"  {col}: unique values now = {sorted(df[col].dropna().unique().tolist())}")

# Verify mortality encoding
for m in ALL_MORT_COLS:
    if m in df.columns:
        rate = df[m].mean() * 100
        print(f"\n{m}: event rate = {rate:.2f}%  (unique = {sorted(df[m].dropna().unique().tolist())})")

Recoded 16 columns from {1,2} → {1,0}:
  Sexo: unique values now = [0, 1]
  O2: unique values now = [0, 1]
  O2 sup.: unique values now = [0, 1]
  Gafas: unique values now = [0, 1]
  Venturi: unique values now = [0, 1]
  Resrv.: unique values now = [0, 1]
  Nebul.: unique values now = [0, 1]
  VNI: unique values now = [0, 1]
  IOT: unique values now = [0, 1]
  VAD: unique values now = [0, 1]
  VM: unique values now = [0, 1]
  MAVA: unique values now = [0, 1]
  TTE: unique values now = [0, 1]
  Mort. 2D: unique values now = [0, 1]
  Mort. 7D: unique values now = [0, 1]
  Mort. 30D: unique values now = [0, 1]

Mort. 2D: event rate = 4.80%  (unique = [0, 1])

Mort. 7D: event rate = 7.24%  (unique = [0, 1])

Mort. 30D: event rate = 10.90%  (unique = [0, 1])


## **11. Mortality outcome monotonicity verification**

The source dataset encodes mortality using three binary indicators — `Mort. 2D`, `Mort. 7D`, and `Mort. 30D`, which follow a **cumulative coding scheme**. Under this scheme, death at an earlier window implies death at all subsequent windows:

| Clinical scenario | Mort. 2D | Mort. 7D | Mort. 30D |
|-------------------|:--------:|:--------:|:---------:|
| Deceased at 0–2 days | 1 | 1 | 1 |
| Deceased at 3–7 days | 0 | 1 | 1 |
| Deceased at 8–30 days | 0 | 0 | 1 |
| Survivor | 0 | 0 | 0 |

> This implies a strict **monotonic nesting constraint**: `Mort. 2D ≤ Mort. 7D ≤ Mort. 30D` for every patient. A violation (e.g., `Mort. 2D = 1` but `Mort. 7D = 0`) indicates either a data-entry error or an encoding inconsistency.

### Verification and correction procedure

1. **Detect violations** — any row where the nesting constraint is broken.
2. **Apply forward-fix** — if `Mort. 2D = 1`, force `Mort. 7D = 1` and `Mort. 30D = 1`; if `Mort. 7D = 1`, force `Mort. 30D = 1`.
3. **Verify** — recount violations after the fix to confirm zero remain.
4. **Report event rates and pattern distribution** before and after correction.

> This correction must be run **after** the binary re-encoding (**Section 9**) so that mortality columns are already in `{0, 1}` format.

> In case of exclusive coding scheme
The outcomes can be change to the other coding scheme (e.g, `Mort. 2D = 1,0,0`).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MORTALITY OUTCOME MONOTONICITY VERIFICATION
# ══════════════════════════════════════════════════════════════════════════════
"""
Task : Verify and enforce cumulative coding of mortality outcomes.
          Under cumulative coding, Mort.2D=1 implies Mort.7D=1 and Mort.30D=1.
Input   : df with binary-recoded mortality columns {0, 1}.
Output  : df with forward-fix applied in-place; violation report printed.
"""

MORTALITY_CODING = "cumulative"   # <- "exclusive" | "cumulative"

m2, m7, m30 = "Mort. 2D", "Mort. 7D", "Mort. 30D"

def _pattern_table(dataframe, m2, m7, m30):
    """Compute mortality pattern counts and percentages from a DataFrame."""
    mask = dataframe[[m2, m7, m30]].notna().all(axis=1)
    sub  = dataframe.loc[mask, [m2, m7, m30]].astype("Int64")
    N    = len(sub)
    tbl  = (sub.value_counts(dropna=False)
               .reset_index(name="n")
               .sort_values("n", ascending=False))
    tbl["pattern (2D-7D-30D)"] = tbl[[m2, m7, m30]].astype("string").agg("-".join, axis=1)
    tbl["%"]                   = (tbl["n"] / N * 100).round(2)
    return tbl[["pattern (2D-7D-30D)", "n", "%"]]

def _event_rates(dataframe, cols):
    """Print event rates for a list of binary outcome columns."""
    for col in cols:
        y     = dataframe[col].astype("Int64")
        valid = y.isin([0, 1])
        rate  = float(y[valid].mean() * 100) if valid.any() else float("nan")
        print(f"  {col}: {rate:.2f}%  (n_events={int(y[valid].sum())}, n_valid={int(valid.sum())})")

if all(c in df.columns for c in [m2, m7, m30]):
    mask_all = df[[m2, m7, m30]].notna().all(axis=1)

    print("=" * 60)
    print("MORTALITY OUTCOME MONOTONICITY VERIFICATION")
    print(f"Coding scheme : {MORTALITY_CODING}")
    print(f"Rows with all 3 endpoints available: {int(mask_all.sum()):,}")
    print("=" * 60)

    # --- Event rates BEFORE correction ---
    print("\nEvent rates BEFORE correction:")
    _event_rates(df, [m2, m7, m30])

    # --- Patterns BEFORE correction ---
    print("\nMortality patterns BEFORE correction:")
    print(_pattern_table(df, m2, m7, m30).to_string(index=False))

    # --- Apply correction ---
    if MORTALITY_CODING == "exclusive":
        sub   = df.loc[mask_all, [m2, m7, m30]].astype("Int64")
        n_multi = int(((sub == 1).sum(axis=1) > 1).sum())
        if n_multi == 0:
            print("\n✅ Mutual exclusivity confirmed. No corrections needed.")
        else:
            print(f"\n⚠️  {n_multi} patients have >1 mortality flag = 1.")
            print("   Review raw data or switch to MORTALITY_CODING = 'cumulative'.")

    elif MORTALITY_CODING == "cumulative":
        # Detect violations BEFORE fix
        viol = (
            ((df[m2] == 1) & (df[m7] == 0)) |
            ((df[m7] == 1) & (df[m30] == 0)) |
            ((df[m2] == 1) & (df[m30] == 0))
        ) & mask_all
        n_viol = int(viol.sum())
        print(f"\nMonotonicity violations (before fix): {n_viol}")

        if n_viol > 0:
            # Apply forward-fix in-place
            df.loc[df[m2] == 1, [m7, m30]] = 1
            df.loc[df[m7] == 1,  m30]      = 1
            print("✅ Forward-fix applied: Mort.2D=1 -> Mort.7D=1, Mort.30D=1.")
        else:
            print("✅ No violations detected. Dataset consistent with cumulative coding.")

        # Verify: recount violations AFTER fix
        mask_all_post = df[[m2, m7, m30]].notna().all(axis=1)
        sub_post      = df.loc[mask_all_post, [m2, m7, m30]].astype("Int64")
        viol_post = (
            ((sub_post[m2] == 1) & (sub_post[m7] == 0)) |
            ((sub_post[m7] == 1) & (sub_post[m30] == 0)) |
            ((sub_post[m2] == 1) & (sub_post[m30] == 0))
        )
        n_viol_post = int(viol_post.sum())
        if n_viol_post == 0:
            print("✅ Verification passed: 0 violations remain after fix.")
        else:
            print(f"⚠️  {n_viol_post} violations still present after fix. Review data.")

    # --- Event rates AFTER correction ---
    print("\nEvent rates AFTER correction:")
    _event_rates(df, [m2, m7, m30])

    # --- Patterns AFTER correction ---
    print("\nMortality patterns AFTER correction:")
    print(_pattern_table(df, m2, m7, m30).to_string(index=False))

else:
    print("⚠️  Not all mortality columns found in df. Correction skipped.")

MORTALITY OUTCOME MONOTONICITY VERIFICATION
Coding scheme : cumulative
Rows with all 3 endpoints available: 2,376

Event rates BEFORE correction:
  Mort. 2D: 4.80%  (n_events=114, n_valid=2376)
  Mort. 7D: 7.24%  (n_events=172, n_valid=2376)
  Mort. 30D: 10.90%  (n_events=259, n_valid=2376)

Mortality patterns BEFORE correction:
pattern (2D-7D-30D)    n     %
              0-0-0 2115 89.02
              1-1-1  113  4.76
              0-0-1   88  3.70
              0-1-1   58  2.44
              0-1-0    1  0.04
              1-0-0    1  0.04

Monotonicity violations (before fix): 2
✅ Forward-fix applied: Mort.2D=1 -> Mort.7D=1, Mort.30D=1.
✅ Verification passed: 0 violations remain after fix.

Event rates AFTER correction:
  Mort. 2D: 4.80%  (n_events=114, n_valid=2376)
  Mort. 7D: 7.28%  (n_events=173, n_valid=2376)
  Mort. 30D: 10.98%  (n_events=261, n_valid=2376)

Mortality patterns AFTER correction:
pattern (2D-7D-30D)    n     %
              0-0-0 2115 89.02
              1-1-1  

## **12. About missing values**

All missing values in the dataset were resolved through column removal in
Sections 6–8:

- **High-missingness columns (>15%)** were removed in **section 8**. These contained
  the bulk of the NaN cells (e.g., PCT 88.8%, Dime.D 78.5%).
- **Hospital and post-ED variables** were removed in **section 8** on clinical grounds
  (data leakage), which also eliminated their associated missingness.

After these removals, the exported `dataset_FINAL.csv` contains **0 NaN
values** across all 33 retained columns. No imputation was required or applied.

## **13. Post-transformation validation**

A final check ensures the cleaned dataset is internally consistent before export:

1. **Duplicate rows or column names:** to visualize structural integrity.
2. **Remaining missingness:** which columns still have NaN values, and at what rate.
3. **Dtype summary:** confirm that numeric columns are numeric and categoricals are not silently stored as `object`.
4. **Attribute reference cross-check:** verify that the retained columns match the expected prehospital variable set from the attribute table.

This is the last opportunity to catch errors before the dataset is consumed by three downstream notebooks. A broken export here propagates silently through the entire pipeline.

In [ ]:
print("=" * 60)
print("POST-TRANSFORMATION VALIDATION")
print("=" * 60)

# 1) Duplicates
dup_rows = df.duplicated().sum()
dup_cols = df.columns.duplicated().sum()
print(f"\n1. Duplicate rows:         {dup_rows}")
print(f"   Duplicate column names:  {dup_cols}")

# 2) Remaining missingness
remaining_miss = (
    df.isna().mean().mul(100).round(2)
    .sort_values(ascending=False)
)
cols_with_miss = remaining_miss[remaining_miss > 0]
print(f"\n2. Columns with remaining NaN: {len(cols_with_miss)} / {df.shape[1]}")
if len(cols_with_miss):
    display(cols_with_miss)

# 3) Dtype summary
print(f"\n3. Dtype distribution:")
display(df.dtypes.value_counts())

# 4) Cross-check with attribute table
try:
    attr_table = pd.read_excel(ATTR_TABLE_PATH)
    expected_cols = attr_table["Atributo"].dropna().str.strip().tolist()

    in_attr_not_df = [c for c in expected_cols if c not in df.columns]
    in_df_not_attr = [c for c in df.columns
                      if c not in expected_cols and c not in ALL_MORT_COLS + ["Fecha"]]

    print(f"\n4. Attribute table cross-check (without outcome variables):")
    print(f"   Expected (attr table):  {len(expected_cols)} variables")
    print(f"   Present in df:          {sum(1 for c in expected_cols if c in df.columns)}")
    if in_attr_not_df:
        print(f"   In attr table but NOT in df: {in_attr_not_df}")
    if in_df_not_attr:
        print(f"   In df but NOT in attr table: {in_df_not_attr}")
except FileNotFoundError:
    print("\n4. Attribute table not found — cross-check skipped.")

print(f"\nFinal shape: {df.shape[0]} rows × {df.shape[1]} columns")
display(df.head(3))

POST-TRANSFORMATION VALIDATION

1. Duplicate rows:         116
   Duplicate column names:  0

2. Columns with remaining NaN: 0 / 33

3. Dtype distribution:


,count
Int64,16
int64,13
float64,4



4. Attribute table cross-check (without outcome variables):
   Expected (attr table):  30 variables
   Present in df:          30

Final shape: 2376 rows × 33 columns


,Edad,Sexo,FR,SpO2,O2,TAS,TAD,TAM,FC,TT,...,Nebul.,VNI,IOT,VAD,VM,MAVA,TTE,Mort. 2D,Mort. 7D,Mort. 30D
0,87,1,23,76,1,82,49,60.000000,88,34.9,...,0,0,0,0,0,0,0,1,1,1
1,87,0,34,85,1,233,126,161.666667,105,36.7,...,1,0,0,0,0,0,0,0,0,0
2,88,1,6,50,1,193,106,135.000000,117,38.2,...,1,0,0,0,0,0,0,0,0,0


## **14. Dataset export**

The cleaned DataFrame (`dataset_FINAL`) is exported as a single CSV file at the path defined in `CLEAN_DATA_PATH`. CSV is preferred over Excel for downstream consumption because:

- It is faster to read/write (no XML overhead).
- It avoids Excel-specific dtype coercion (e.g., dates, leading zeros).
- It is natively supported by `pd.read_csv()` in **NB2**–**NB4**.

The export uses `index=False` to avoid writing the pandas row index as an extra column.

### Output

| File | Format | Description |
|------|--------|-------------|
| `dataset_FINAL.csv` | CSV (UTF-8) | Definitive cleaned dataset for the entire pipeline |


> A single, stable output path (not timestamped) ensures that **NB2**, **NB3**, and **NB4** always load the same file. If a new version is needed, re-running this notebook overwrites the previous export, version control is handled by git or Drive revision history, not by filename proliferation.

In [ ]:
df.to_csv(CLEAN_DATA_PATH, index=False)

file_size_kb = os.path.getsize(CLEAN_DATA_PATH) / 1024
print(f"✅ Export completed: {CLEAN_DATA_PATH}")
print(f"   Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"   File size: {file_size_kb:.0f} KB")

✅ Export completed: /content/drive/MyDrive/Colab Notebooks/TFG/dataset_FINAL.csv
   Shape: 2376 rows × 33 columns
   File size: 223 KB


## **15. Conclusions**

### What was achieved

1. **Structural cleaning:** column names standardised, ghost columns removed, missing-value tokens normalised.
2. **Missingness quantification:** per-attribute and per-instance analysis, with a 15 % threshold applied programmatically to flag high-missingness columns.
3. **Column removal:** 58 columns removed across three categories (high missingness, hospital/post-ED leakage, non-informative text), reducing the dataset to the prehospital feature set defined in the attribute reference table.
4. **Binary re-encoding:** clinical `{1, 2}` convention converted to modelling-standard `{1, 0}` for all binary variables and mortality outcomes.
5. **Mortality monotonicity verification:** cumulative coding constraint (`Mort. 2D ≤ Mort. 7D ≤ Mort. 30D`) verified and forward-fix applied to any violations.
6. **Duplicate detection:** duplicated rows identified; retained to preserve statistical power in low-prevalence outcomes.
7. **Single export:** one CSV file (`dataset_FINAL.csv`) that serves as the sole input for **NB2**, **NB3**, and **NB4**.

### What was deferred

- **Imputation:** not applied. All high-missingness columns were removed; remaining gaps are MAR/MNAR and handled in downstream modelling.
- **Outlier detection:** addressed in **NB3** with clinical plausibility ranges.
- **Feature selection and modelling:** addressed in **NB4** (VIF, logistic regression, risk scoring).

---